# Lab 4 · SLIM

The runnable companion of the lab page, `docs/labs/04-slim.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "slim"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
print(model.fit_info)  # how many non-zero weights SLIM kept
item = int(data.item_pop.argmax())
lab.neighbours(model, data, item)

**Exercise.** How does the number of non-zero weights change with `slim_alpha`? Fit at 0.0001, 0.001 and 0.01 and print `fit_info['nonzeros']` each time.

<details><summary>Hint</summary>

Loop over the three values; change only `slim_alpha` with `{**best, "slim_alpha": a}`.

</details>

<details><summary>Solution</summary>

```python
for a in (0.0001, 0.001, 0.01):
    m = lab.fit(METHOD, data, **{**best, "slim_alpha": a})
    print(a, m.fit_info["nonzeros"])
```
The L1 part of the penalty sets more weights exactly to zero as alpha grows: the count falls steeply.

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "slim_alpha", [1e-05, 0.0001, 0.001, 0.01, 0.1], start="best")
lab.plot_sweep(table)
table

**Your prediction for `slim_alpha`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "slim_max_iter", [50, 100, 500], start="best")
lab.plot_sweep(table)
table

**Your prediction for `slim_max_iter`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh slim:more-neighbours
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `more-neighbours` is ready to run; `min-count`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_slim_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:more-neighbours")

In [ ]:
lab.compare(METHOD, f"{METHOD}:min-count")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`negative-weights` in `labs/04-slim/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:negative-weights")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:negative-weights", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.